# DINOv2 寵物分類教學 / DINOv2 for Pet Classification

從 **直覺** → **解剖** → **應用** → **視覺證據** → **收束**，一份完整的 self-supervised ViT 教學。

## 我們會做什麼？

- 載入 Meta 預訓練的 **DINOv2-with-registers-small**（ViT-S/14, 21M 參數）
- 在 **Oxford-IIIT Pet**（37 種貓狗品種）做 **kNN 分類**與 **Linear Probe**
- 視覺化 DINO 招牌的 **attention map 自動 segmentation**
- 對比有 / 無 **register tokens** 的差異

## 教學結構

| Part | 章節 | 目的 |
|---|---|---|
| **I. 直覺** | Ch 1–3 | 理解 DINO 是什麼、為何重要 |
| **II. 解剖** | Ch 4–5 | 在「用」之前，先「看懂」模型內部 |
| **III. 應用** | Ch 6–10 | 用 DINOv2 做下游分類，量化 feature 品質 |
| **IV. 證據** | Ch 11–12 | 視覺化 attention map，印證 Part I 直覺 |
| **V. 收束** | Ch 13 | 總結、延伸方向 |

---
# Part I — 直覺 / Intuition

> 在動手寫 code 之前，先理解 **DINO 在做什麼**、**為什麼這樣設計**。沒有這層理解，後面看到 attention map 自動畫出貓的輪廓你會嚇到不知道為什麼。

## 第 1 章：DINO 在解什麼問題？

### Self-Supervised Learning (SSL) 的動機

在 vision 領域，**標註資料昂貴**：
- ImageNet 有 1,400 萬張標註圖
- 但網路上有 **數十億張** 沒標註的圖
- SSL 的問題意識：「能不能不用標籤，也讓模型學到好的視覺特徵？」

### 圖像 SSL 為何難？

NLP 有天然的 SSL 任務：「猜下一個 token」。
但圖像沒有這樣明確的「next」結構，所以歷史上嘗試過很多方法：

| 方法 | 思路 | 問題 |
|---|---|---|
| Autoencoder | 重建像素 | 強迫保留低階紋理，learnt features 不夠抽象 |
| Rotation prediction | 預測圖被旋轉幾度 | 任務太簡單，feature 不夠強 |
| Contrastive (SimCLR/MoCo) | 同張圖的不同 augmentation 拉近、其他推遠 | 需要 large negative batch、靠 batch size 撐效能 |
| **DINO (2021)** | **Self-distillation：student 學 teacher 的輸出** | **不需要 negatives、不需要標籤、attention 自動 segment 物體** |

### DINO 的一句話定位

> **「不用標籤的 ViT 預訓練，得到的 features 強到 frozen 後直接做分類就贏；更神奇的是 attention map 會自動分割出物體輪廓。」**

這份教學會親自驗證最後這兩個 claim。

## 第 2 章：DINO 演算法的核心思想

> 我們是用 pretrained model，**不在這章寫 training code**。但理解原理才能解釋後面為什麼 attention map 會自動 segment 物體。

### 兩個關鍵組件

**(1) Student-Teacher 架構**

兩個結構一模一樣的 ViT：
- **Student** 用 SGD/AdamW 更新
- **Teacher** 不直接訓練，而是用 student 參數的 **指數移動平均 (EMA)**：
  $$\theta_{teacher} \leftarrow \tau \cdot \theta_{teacher} + (1-\tau) \cdot \theta_{student}$$
- Teacher 是「比較慢、比較穩定」的 student

**(2) Multi-crop 策略**

每張圖隨機切出：
- **2 個 global crop**（大區域，~224×224）
- **多個 local crop**（小區域，~96×96）

規則：
- **Teacher 只看 global crop**
- **Student 看所有 crop**
- 訓練目標：student 對 local crop 的輸出 ≈ teacher 對 global crop 的輸出
- 直覺：「**看到局部要能推測出整體**」→ 強迫模型學到語意而非像素細節

### 為什麼不會 collapse？

最大的擔憂：兩個一樣的模型互相學，可能所有 input 都輸出同一個向量（trivial solution）。
DINO 用兩招防止：

- **Centering**：teacher 輸出減掉一個移動平均的中心（避免某個維度永遠 dominant）
- **Sharpening**：teacher 用較低 softmax temperature（讓 teacher 輸出比較尖銳、有資訊量）

下方用一張概念圖把整個流程串起來。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import numpy as np

# 支援繁體中文字型 / Enable Traditional Chinese font
plt.rcParams['font.family'] = ['Microsoft JhengHei', 'PingFang TC', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('DINO 自監督訓練概念圖 / DINO Self-Distillation Flow', fontsize=14, pad=10)

# 原始圖像
img_box = mpatches.FancyBboxPatch((0.3, 3.3), 1.6, 1.6, boxstyle='round,pad=0.05',
                                   facecolor='#F4D03F', edgecolor='#7D6608', linewidth=2)
ax.add_patch(img_box)
ax.text(1.1, 4.1, '原圖\nImage', ha='center', va='center', fontsize=10, fontweight='bold')

# Multi-crop 區
ax.annotate('', xy=(3.0, 5.5), xytext=(2.0, 4.5),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#444'))
ax.annotate('', xy=(3.0, 2.5), xytext=(2.0, 3.7),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#444'))
ax.text(2.5, 4.7, 'multi-crop', fontsize=8, color='#555', style='italic')

# Global crops (teacher 看)
for i, y in enumerate([5.8, 4.8]):
    box = mpatches.FancyBboxPatch((3.1, y-0.4), 1.2, 0.8, boxstyle='round,pad=0.03',
                                   facecolor='#AED6F1', edgecolor='#1B4F72', linewidth=1.5)
    ax.add_patch(box)
    ax.text(3.7, y, f'global\ncrop {i+1}', ha='center', va='center', fontsize=8)

# Local crops (student 看更多)
for i, y in enumerate([3.0, 2.2, 1.4]):
    box = mpatches.FancyBboxPatch((3.3, y-0.3), 0.8, 0.6, boxstyle='round,pad=0.03',
                                   facecolor='#FADBD8', edgecolor='#922B21', linewidth=1.2)
    ax.add_patch(box)
    ax.text(3.7, y, f'local {i+1}', ha='center', va='center', fontsize=7)

# Teacher 模型框
teacher_box = mpatches.FancyBboxPatch((5.0, 4.5), 2.2, 1.7, boxstyle='round,pad=0.1',
                                       facecolor='#D6EAF8', edgecolor='#1B4F72', linewidth=2.5)
ax.add_patch(teacher_box)
ax.text(6.1, 5.7, 'Teacher ViT', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(6.1, 5.1, '(EMA of student)\n不接收梯度', ha='center', va='center', fontsize=8, color='#555')

# Student 模型框
student_box = mpatches.FancyBboxPatch((5.0, 1.0), 2.2, 1.7, boxstyle='round,pad=0.1',
                                       facecolor='#FADBD8', edgecolor='#922B21', linewidth=2.5)
ax.add_patch(student_box)
ax.text(6.1, 2.2, 'Student ViT', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(6.1, 1.6, '(SGD 更新)', ha='center', va='center', fontsize=8, color='#555')

# Crops → models 連接
ax.annotate('', xy=(5.0, 5.3), xytext=(4.3, 5.3), arrowprops=dict(arrowstyle='->', lw=1.2, color='#1B4F72'))
for y in [3.0, 2.2, 1.4]:
    ax.annotate('', xy=(5.0, 1.85), xytext=(4.1, y), arrowprops=dict(arrowstyle='->', lw=0.8, color='#922B21', alpha=0.5))
ax.annotate('', xy=(5.0, 1.85), xytext=(4.3, 5.3), arrowprops=dict(arrowstyle='->', lw=0.8, color='#922B21', alpha=0.5))

# Centering + Sharpening on teacher output
ax.text(7.4, 5.4, '→  centering\n   + sharpen', ha='left', va='center', fontsize=8.5, color='#1B4F72')

# Loss
loss_box = mpatches.FancyBboxPatch((9.6, 3.0), 1.8, 1.2, boxstyle='round,pad=0.1',
                                    facecolor='#D5F5E3', edgecolor='#1E8449', linewidth=2)
ax.add_patch(loss_box)
ax.text(10.5, 3.6, 'Cross-Entropy\nLoss\n(matching\ndistributions)', ha='center', va='center', fontsize=8)

# Outputs → loss
ax.annotate('', xy=(9.6, 4.0), xytext=(8.6, 5.0), arrowprops=dict(arrowstyle='->', lw=1.2, color='#1B4F72'))
ax.annotate('', xy=(9.6, 3.2), xytext=(8.6, 1.9), arrowprops=dict(arrowstyle='->', lw=1.2, color='#922B21'))

# Gradient back to student
ax.annotate('', xy=(7.2, 1.5), xytext=(10.5, 2.9),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#7D3C98', connectionstyle='arc3,rad=0.3'))
ax.text(9.0, 1.6, '反向傳播\nbackprop', ha='center', va='center', fontsize=8, color='#7D3C98')

# EMA update
ax.annotate('', xy=(6.1, 4.5), xytext=(6.1, 2.7),
            arrowprops=dict(arrowstyle='->', lw=1.8, color='#117864', linestyle='dashed'))
ax.text(6.5, 3.6, 'EMA update\n(no gradient)', ha='left', va='center', fontsize=8.5, color='#117864', style='italic')

# Right side note
ax.text(12.0, 6.0, '關鍵 take-away:', fontsize=10, fontweight='bold', color='#444')
ax.text(12.0, 5.4, '• Student 學會\n  「從局部推全局」', fontsize=8.5, color='#444', va='top')
ax.text(12.0, 4.3, '• Teacher 是 student\n  自己的「穩定版」', fontsize=8.5, color='#444', va='top')
ax.text(12.0, 3.2, '• 沒有標籤、沒有\n  negatives', fontsize=8.5, color='#444', va='top')
ax.text(12.0, 2.1, '• Attention 學會\n  尋找「語意主體」', fontsize=8.5, color='#444', va='top')

plt.tight_layout(); plt.show()

## 第 3 章：DINOv1 → v2 → v3 的演進

| 維度 | DINOv1 (2021) | **DINOv2 (2023)** ⭐ | DINOv3 (2025) |
|---|---|---|---|
| Backbone | ViT-S/16, ViT-B/16 | ViT-S/14 ~ ViT-g/14 | ViT-S/16 ~ ViT-7B/16, ConvNeXt |
| 預訓練資料 | ImageNet (1.4M) | LVD-142M（自動 curated 142M） | LVD-1.6B |
| Patch size | 16 | **14**（更細） | 16 |
| 訓練技巧 | 原版 self-distillation | + iBOT（masked patch prediction）、KoLeo regularizer | + Gram anchoring、gated attention |
| Register tokens | ❌ | ✅（with-registers 版本） | ✅ |

### 為什麼 DINOv2 需要 register tokens？

論文 ["Vision Transformers Need Registers" (Darcet et al., 2023)](https://arxiv.org/abs/2309.16588) 發現：

- 訓練好的 DINOv2 attention map 上偶爾出現 **高激活的雜訊點**（artifact）
- 原因：模型沒有「臨時計算空間」，被迫把全域計算暫存到不重要的 patch token 上 → 那些 patch 的 attention 變得異常
- 解法：加 4 個 **可學習的 register token** 當作 scratchpad
- 結果：attention map 變乾淨、下游任務也微幅提升

**Part IV 會親眼看到這個差異。**

### 我們的選擇：`facebook/dinov2-with-registers-small`

- 21M 參數（ViT-S/14），在 MPS / CPU 上跑得動
- 帶 register tokens 的最新最佳實踐版本
- embedding dim = 384，剛好夠做下游分類

---
# Part II — 解剖 / Anatomy

> 在「用」之前先「看懂」。很多教學跳過這步直接呼叫 model，讀者永遠不知道 `outputs.last_hidden_state` 裡每個 token 是什麼。我們拆得清清楚楚再往下走。

## 第 4 章：載入 DINOv2 並拆解結構

### 環境安裝

我們用 HuggingFace `transformers` 載入模型——比 `torch.hub` 更穩定、有 register-token 版本。

In [ ]:
%pip install transformers torchvision scikit-learn --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from transformers import AutoModel, AutoImageProcessor
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.family'] = ['Microsoft JhengHei', 'PingFang TC', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 自動偵測裝置 / Auto-detect device
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'使用裝置 / Device: {DEVICE}')

In [ ]:
MODEL_NAME = 'facebook/dinov2-with-registers-small'

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
# 注意：attn_implementation='eager' 是為了 Part IV 能拿到 attention weights
# 預設的 SDPA 較快但不回傳 attention（transformers >=4.4x 預設用 SDPA）
model     = AutoModel.from_pretrained(MODEL_NAME, attn_implementation='eager').to(DEVICE).eval()

# 統計參數量
n_params = sum(p.numel() for p in model.parameters())
print(f'模型: {MODEL_NAME}')
print(f'參數量 / Params: {n_params/1e6:.2f}M')
print(f'隱藏維度 / Hidden dim: {model.config.hidden_size}')
print(f'Patch size: {model.config.patch_size}')
print(f'Transformer 層數 / # layers: {model.config.num_hidden_layers}')
print(f'Attention heads: {model.config.num_attention_heads}')
print(f'Register tokens: {model.config.num_register_tokens}')
print(f'Attention implementation: {model.config._attn_implementation}')
print(f'\n預訓練時的影像尺寸 / Pretrain image size: {model.config.image_size}')

In [ ]:
# 印出模型骨架（摺疊顯示）/ Print model skeleton
print('── DINOv2-with-registers-small 結構 ──\n')
print('Embeddings (patch projection + CLS + 4 registers + pos embed):')
for name, mod in model.embeddings.named_children():
    print(f'  {name:25s} {mod.__class__.__name__}')

print(f'\nEncoder: {len(model.encoder.layer)} × Transformer Block')
block0 = model.encoder.layer[0]
for name, _ in block0.named_children():
    print(f'  Block.{name}')

print('\nLayernorm (最終輸出前) / Final LayerNorm:')
print(f'  {model.layernorm}')

**讀懂這個結構：**

- `embeddings` 把 `224×224×3` 影像切成 `16×16=256` 個 patch，每塊投影到 384 維，前面再串上 `[CLS]` 和 4 個 register tokens → 共 **261 個 token**
- `encoder` 是 12 層 Transformer Block（每層含 multi-head attention + MLP + LayerNorm）
- 最終輸出的 `last_hidden_state` shape 為 `(batch, 261, 384)`

下一章我們會跑一張圖、把 261 個 token 看清楚。

## 第 5 章：一張圖的完整 Forward Pass

用一張測試圖（隨便一張寵物圖），把整條 pipeline 從輸入跑到輸出，**看清楚每個 tensor shape 在說什麼**。

In [ ]:
# 先預載一張測試影像（用 dataset 第一張）
# 這裡 download=True 第一次跑會花一點時間下載資料集
DATA_ROOT = './data'
_preview_ds = OxfordIIITPet(root=DATA_ROOT, split='trainval', download=True)
sample_img, sample_label = _preview_ds[0]
print(f'樣本標籤 / Label: {_preview_ds.classes[sample_label]}')
print(f'原圖尺寸 / Image size: {sample_img.size}')
sample_img

In [ ]:
# 用 processor 把 PIL → tensor（含 resize、normalize）
inputs = processor(images=sample_img, return_tensors='pt').to(DEVICE)
print(f'pixel_values shape: {inputs["pixel_values"].shape}  (B, C, H, W)')

with torch.no_grad():
    outputs = model(**inputs)

hidden = outputs.last_hidden_state
print(f'last_hidden_state shape: {hidden.shape}  (B, num_tokens, dim)')

# 拆解 token 序列
B, N, D = hidden.shape
n_reg = model.config.num_register_tokens
n_patches = N - 1 - n_reg
grid = int(n_patches ** 0.5)
print(f'\n── Token 拆解 / Token Breakdown ──')
print(f'  Token 0      : [CLS]            shape={hidden[:, 0].shape}')
print(f'  Token 1..{n_reg}    : Registers (×{n_reg})   shape={hidden[:, 1:1+n_reg].shape}')
print(f'  Token {1+n_reg}..{N-1} : Patches (×{n_patches} = {grid}×{grid} grid) shape={hidden[:, 1+n_reg:].shape}')
print(f'  Embedding dim: {D}')

In [ ]:
# 視覺化：token 序列構成 / Visualize the token sequence layout
fig, ax = plt.subplots(figsize=(14, 2.5))
ax.set_xlim(0, N); ax.set_ylim(0, 1); ax.axis('off')

# CLS
ax.add_patch(mpatches.Rectangle((0, 0), 1, 1, facecolor='#F4D03F', edgecolor='black'))
ax.text(0.5, 0.5, 'CLS', ha='center', va='center', fontsize=8, fontweight='bold')
# Registers
for i in range(n_reg):
    ax.add_patch(mpatches.Rectangle((1+i, 0), 1, 1, facecolor='#D6EAF8', edgecolor='black'))
    ax.text(1.5+i, 0.5, f'R{i+1}', ha='center', va='center', fontsize=7)
# Patches (just show first/last few + ellipsis)
patch_start = 1 + n_reg
for i in range(5):
    ax.add_patch(mpatches.Rectangle((patch_start+i, 0), 1, 1, facecolor='#ABEBC6', edgecolor='black'))
    ax.text(patch_start+i+0.5, 0.5, f'p{i+1}', ha='center', va='center', fontsize=6)
ax.text(patch_start+5+10, 0.5, '...', ha='center', va='center', fontsize=20)
for i in range(5):
    idx = N - 5 + i
    ax.add_patch(mpatches.Rectangle((idx, 0), 1, 1, facecolor='#ABEBC6', edgecolor='black'))
    ax.text(idx+0.5, 0.5, f'p{n_patches-4+i}', ha='center', va='center', fontsize=6)

ax.text(0.5, -0.4, 'global', ha='center', fontsize=9, color='#7D6608')
ax.text(1+n_reg/2, -0.4, f'registers (×{n_reg})\nscratchpad', ha='center', fontsize=9, color='#1B4F72')
ax.text((patch_start + N)/2, -0.4, f'patches (×{n_patches}, 對應 {grid}×{grid} 空間位置)', ha='center', fontsize=9, color='#196F3D')
ax.set_title('DINOv2-with-registers token 序列 / Token sequence layout', fontsize=11)
plt.tight_layout(); plt.show()

print(f'\n各 token 的用途：')
print(f'• [CLS]      → 全圖摘要，常用做 image-level feature（分類）')
print(f'• Registers  → 模型「計算暫存」，不對應空間位置，不該用於 dense task')
print(f'• Patches    → 對應原圖 14×14 像素區塊，可重排成空間 grid 做 segmentation/attention map')

---
# Part III — 應用 / Application

> 把抽象的「好 features」變成 **可量化的下游效能**。先 kNN 給驚奇感，再 linear probe 給嚴謹感。

## 第 6 章：Oxford-IIIT Pet 資料準備

- 37 個類別（25 種狗 + 12 種貓品種）
- 約 7,349 張影像（train 3,680 / test 3,669）
- 細粒度分類：分辨「英短 vs 俄羅斯藍」、「貴賓 vs 比熊」這種等級

為何選這個資料集？粗類別任何 backbone 都會分對，**細粒度才看得出 features 是否真的有語意**。

In [ ]:
# DINOv2 預訓練的 normalize 統計（ImageNet 標準）
DINO_MEAN = [0.485, 0.456, 0.406]
DINO_STD  = [0.229, 0.224, 0.225]

# 224x224：給 feature extraction（與預訓練設定一致）
transform_224 = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=DINO_MEAN, std=DINO_STD),
])

train_ds = OxfordIIITPet(root=DATA_ROOT, split='trainval', download=True, transform=transform_224)
test_ds  = OxfordIIITPet(root=DATA_ROOT, split='test',     download=True, transform=transform_224)
NUM_CLASSES = len(train_ds.classes)

print(f'類別數 / Classes: {NUM_CLASSES}')
print(f'訓練樣本 / Train: {len(train_ds)}')
print(f'測試樣本 / Test : {len(test_ds)}')
print(f'前 5 類: {train_ds.classes[:5]} ...')

In [ ]:
# 視覺化 8 張樣本 / Visualize 8 random samples
# 為了顯示原圖（不帶 normalize），另開一個 dataset
_vis_ds = OxfordIIITPet(root=DATA_ROOT, split='trainval', download=False)

idxs = np.random.RandomState(42).choice(len(_vis_ds), 8, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.flat, idxs):
    img, lbl = _vis_ds[idx]
    ax.imshow(img)
    ax.set_title(_vis_ds.classes[lbl], fontsize=9)
    ax.axis('off')
plt.suptitle('Oxford-IIIT Pet 樣本 / Random samples', fontsize=12)
plt.tight_layout(); plt.show()

## 第 7 章：Feature Extraction

把整個 train + test 跑過 DINOv2 一次，把每張圖變成一個 **384 維向量**。之後 kNN 和 linear probe 都用這些 cached features，**不用再過 backbone**——這就是 frozen-backbone evaluation 的標準做法。

### Pooling 策略決策

從 261 個 token 怎麼變成 1 個 image-level feature？有幾種選擇：

| 策略 | 公式 | 用途 |
|---|---|---|
| **CLS only** ⭐ | `hidden[:, 0]` | 標準做法，DINOv2 paper 用這個 |
| **Mean of patches** | `hidden[:, 5:].mean(dim=1)` | 對 dense feature 任務（detection、seg）較好 |
| **CLS + Mean patches** | `concat([cls, mean_patches])` | 兩者並用，dim 變 2× |

我們用 **CLS only**（簡單、與論文一致）。下方函式裡你可以隨時切換 `pool_strategy` 對比。

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def extract_features(dataset, model, device, batch_size=64, pool_strategy='cls'):
    """提取整個 dataset 的 features。
    
    pool_strategy:
      'cls'        - 用 [CLS] token (預設)
      'mean_patch' - 對 patch tokens 取平均（跳過 registers）
      'cls_mean'   - CLS 與 mean patches 串接 (dim 變 2x)
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    feats, labels = [], []
    n_reg = model.config.num_register_tokens
    patch_start = 1 + n_reg
    for x, y in tqdm(loader, desc=f'extracting ({pool_strategy})'):
        x = x.to(device)
        out = model(pixel_values=x).last_hidden_state  # (B, N, D)
        if pool_strategy == 'cls':
            f = out[:, 0]
        elif pool_strategy == 'mean_patch':
            f = out[:, patch_start:].mean(dim=1)
        elif pool_strategy == 'cls_mean':
            f = torch.cat([out[:, 0], out[:, patch_start:].mean(dim=1)], dim=-1)
        else:
            raise ValueError(pool_strategy)
        feats.append(f.cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)

print('開始提取 features (一次性，後面 kNN/probe 重用)...')
train_feats, train_labels = extract_features(train_ds, model, DEVICE, pool_strategy='cls')
test_feats,  test_labels  = extract_features(test_ds,  model, DEVICE, pool_strategy='cls')
print(f'\ntrain_feats shape: {train_feats.shape}')
print(f'test_feats  shape: {test_feats.shape}')

## 第 8 章：kNN 分類（零訓練）— WOW 時刻

**完全不訓練**，只把 test feature 拿去 train features 裡找最近鄰、投票決定類別。

如果 features 真的有語意，這個簡單到不行的演算法應該就能跑出體面的 accuracy。

### 為什麼用 cosine 距離？

DINO 訓練時用的 loss 等價於要 features 在 **單位球面** 上對齊，所以 cosine similarity（= 球面距離）比 Euclidean 更貼近 feature space 的幾何。

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Normalize features → cosine 等價於 dot product
train_feats_n = F.normalize(train_feats, dim=-1).numpy()
test_feats_n  = F.normalize(test_feats,  dim=-1).numpy()

knn = KNeighborsClassifier(n_neighbors=20, metric='cosine', n_jobs=-1)
knn.fit(train_feats_n, train_labels.numpy())
knn_preds = knn.predict(test_feats_n)
knn_acc = (knn_preds == test_labels.numpy()).mean()
print(f'kNN (k=20, cosine) 測試準確率 / Test accuracy: {knn_acc*100:.2f}%')
print(f'（完全沒做任何梯度更新！純粹靠 DINOv2 features 的可分性）')

### Nearest neighbor 視覺化

kNN 不只是個 accuracy 數字——把鄰居印出來，可以**親眼看到 feature space 真的有語意**：同品種的貓狗在 feature 空間裡是相鄰的。

In [ ]:
# 取幾張 query 影像，顯示其在 train set 中的 top-5 最近鄰
rng = np.random.RandomState(7)
query_idxs = rng.choice(len(test_ds), 4, replace=False)

# 用矩陣乘法找 nearest neighbor（cosine on normalized features）
q = torch.tensor(test_feats_n[query_idxs])
t = torch.tensor(train_feats_n)
sim = q @ t.T  # (4, N_train)
topk = sim.topk(5, dim=-1).indices.numpy()

vis_test  = OxfordIIITPet(root=DATA_ROOT, split='test',     download=False)
vis_train = OxfordIIITPet(root=DATA_ROOT, split='trainval', download=False)

fig, axes = plt.subplots(4, 6, figsize=(16, 11))
for row, qi in enumerate(query_idxs):
    qimg, qlbl = vis_test[qi]
    axes[row, 0].imshow(qimg)
    axes[row, 0].set_title(f'Query:\n{vis_test.classes[qlbl]}', fontsize=9, color='#922B21', fontweight='bold')
    axes[row, 0].axis('off')
    for col, ni in enumerate(topk[row]):
        nimg, nlbl = vis_train[ni]
        axes[row, col+1].imshow(nimg)
        match = '✓' if nlbl == qlbl else '✗'
        color = '#1E8449' if nlbl == qlbl else '#A93226'
        axes[row, col+1].set_title(f'NN {col+1} {match}\n{vis_train.classes[nlbl]}', fontsize=8, color=color)
        axes[row, col+1].axis('off')
plt.suptitle('kNN 最近鄰視覺化（feature space 中與 query 最近的訓練圖）', fontsize=12)
plt.tight_layout(); plt.show()

## 第 9 章：Linear Probe — SSL 領域的標準考試

**Linear Probe 協議**：凍結整個 backbone，**只訓練最後一層 `nn.Linear`** 把 feature 映射到類別。

為何這是標準？
- 把「feature 品質」和「fine-tune 能力」解耦——你考的就是 feature 本身好不好用
- 如果 features 真的線性可分，一層 linear 就夠
- DINO、SimCLR、MAE 全用這個指標——大家都用同一把尺，可橫向比較

### 我們已經提取好 features 了，所以 linear probe = 在 (N, 384) 矩陣上訓一個 logistic regression

In [ ]:
from torch.utils.data import TensorDataset

# 不用 normalize 的 features（linear layer 自己會學 scale）
train_x = train_feats.to(DEVICE)
train_y = train_labels.to(DEVICE)
test_x  = test_feats.to(DEVICE)
test_y  = test_labels.to(DEVICE)

probe = nn.Linear(train_feats.shape[1], NUM_CLASSES).to(DEVICE)
optim_ = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 50
BATCH  = 128
N_train = train_x.size(0)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim_, T_max=EPOCHS)
history = {'train_loss': [], 'test_acc': []}

for epoch in range(EPOCHS):
    probe.train()
    perm = torch.randperm(N_train, device=DEVICE)
    losses = []
    for i in range(0, N_train, BATCH):
        idx = perm[i:i+BATCH]
        logits = probe(train_x[idx])
        loss = F.cross_entropy(logits, train_y[idx])
        optim_.zero_grad(); loss.backward(); optim_.step()
        losses.append(loss.item())
    scheduler.step()

    probe.eval()
    with torch.no_grad():
        test_preds = probe(test_x).argmax(dim=-1)
        test_acc = (test_preds == test_y).float().mean().item()
    history['train_loss'].append(np.mean(losses))
    history['test_acc'].append(test_acc)
    if (epoch+1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}  loss={np.mean(losses):.4f}  test_acc={test_acc*100:.2f}%')

probe_acc = history['test_acc'][-1]
print(f'\nFinal Linear Probe 測試準確率 / Test accuracy: {probe_acc*100:.2f}%')

In [ ]:
# 訓練曲線 / Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history['train_loss'], color='#922B21')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss')
axes[0].set_title('訓練 Loss / Training loss'); axes[0].grid(alpha=0.3)
axes[1].plot([a*100 for a in history['test_acc']], color='#1E8449')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Test Acc (%)')
axes[1].set_title('測試準確率 / Test accuracy'); axes[1].grid(alpha=0.3)
axes[1].axhline(knn_acc*100, color='#7D3C98', linestyle='--', label=f'kNN baseline ({knn_acc*100:.2f}%)')
axes[1].legend()
plt.tight_layout(); plt.show()

## 第 10 章：kNN vs Linear Probe 對比分析

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# 左：bar chart
methods = ['kNN\n(0 訓練)', 'Linear Probe\n(僅一層 fc)']
accs    = [knn_acc*100, probe_acc*100]
colors  = ['#7D3C98', '#1E8449']
bars = axes[0].bar(methods, accs, color=colors, width=0.55)
for b, a in zip(bars, accs):
    axes[0].text(b.get_x()+b.get_width()/2, a+0.5, f'{a:.2f}%',
                 ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_ylim(0, max(accs)*1.15)
axes[0].set_title('DINOv2 frozen features 下游效能對比', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# 右：linear probe 的 confusion matrix
with torch.no_grad():
    test_preds = probe(test_x).argmax(dim=-1).cpu().numpy()
cm = confusion_matrix(test_labels.numpy(), test_preds)
im = axes[1].imshow(cm, cmap='Blues')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].set_title(f'Linear Probe Confusion Matrix\n({NUM_CLASSES} 類)', fontsize=11)
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout(); plt.show()

# 找出最常被混淆的類別 pair
np.fill_diagonal(cm, 0)
top_confusion = np.unravel_index(np.argsort(cm.ravel())[-5:][::-1], cm.shape)
print('最常被混淆的 5 對類別 / Top-5 confused pairs:')
for t, p in zip(*top_confusion):
    print(f'  「{train_ds.classes[t]}」  被預測成  「{train_ds.classes[p]}」  ({cm[t,p]} 次)')

### 對比 takeaway

- **kNN 已經很強**：完全沒訓練、靠 cosine 距離就拿到體面 accuracy → 直接證明 DINOv2 features 有強烈的「類別語意可分性」
- **Linear Probe 通常再多幾趴**：一層 linear 把 cosine 距離換成「學到的線性分界」，在邊界 case 上更精確
- **混淆模式有趣**：常常是視覺極相似的品種互混（例如 Egyptian Mau ↔ Bengal cat），說明 feature 確實在分「視覺差異」而不只是表面顏色

---
# Part IV — 視覺證據 / Evidence

> Part III 用數字證明 features 好。現在我們**用肉眼看見「為什麼好」**——回頭呼應 Part I 的直覺。

## 第 11 章：Attention Map 視覺化（DINO 招牌）

DINO 系列最令人驚艷的「湧現現象」：

> 沒有任何分割標註訓練，[CLS] token 對 patch tokens 的 attention 卻會**自動勾勒出物體輪廓**。

為什麼？回想 Part I 第 2 章：
- Student 看 local crop，要預測 teacher 看 global crop 的輸出
- 為了做到這件事，模型必須學會「從局部認出整體」
- → [CLS] token 必須**聚焦在主體物件上**，才能彙整足夠語意
- → attention map 自然強調主體

### 設定：用 518×518 高解析度跑

518 = 14 × 37 → 產生 37×37 = 1369 patches，attention map 更細緻。
`transformers` 透過 `interpolate_pos_encoding=True` 自動把 position embedding 插值到對應 grid。

In [ ]:
ATTN_SIZE = 518
attn_transform = transforms.Compose([
    transforms.Resize((ATTN_SIZE, ATTN_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=DINO_MEAN, std=DINO_STD),
])

def get_attention_map(pil_img, model, device, layer=-1):
    """取最後一層、CLS 對所有 patch 的 attention（平均所有 heads 或保留分頭）。
    回傳: (heads, grid, grid) numpy array
    """
    x = attn_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(pixel_values=x, output_attentions=True, interpolate_pos_encoding=True)
    attn = out.attentions[layer][0]  # (heads, N, N)
    n_reg = getattr(model.config, 'num_register_tokens', 0)
    patch_start = 1 + n_reg
    cls_to_patch = attn[:, 0, patch_start:]  # (heads, num_patches)
    n_patches = cls_to_patch.shape[-1]
    grid = int(n_patches ** 0.5)
    return cls_to_patch.reshape(-1, grid, grid).cpu().numpy()

# 拿一張寵物原圖
pet_img, pet_lbl = _vis_ds[0]
attn_heads = get_attention_map(pet_img, model, DEVICE)
print(f'attention map shape: {attn_heads.shape}  (heads, grid, grid)')

In [ ]:
# 多頭視覺化 / Visualize attention per head
n_heads = attn_heads.shape[0]
fig, axes = plt.subplots(2, n_heads + 1, figsize=(2.2*(n_heads+1), 5))

# 第一欄：原圖 + 平均 attention overlay
axes[0,0].imshow(pet_img.resize((ATTN_SIZE, ATTN_SIZE)))
axes[0,0].set_title(f'原圖\n{_vis_ds.classes[pet_lbl]}', fontsize=9)
axes[0,0].axis('off')
mean_attn = attn_heads.mean(0)
axes[1,0].imshow(pet_img.resize((ATTN_SIZE, ATTN_SIZE)))
ax_overlay = axes[1,0].imshow(
    np.kron(mean_attn, np.ones((ATTN_SIZE//mean_attn.shape[0], ATTN_SIZE//mean_attn.shape[1]))),
    cmap='jet', alpha=0.5)
axes[1,0].set_title('平均 attention\n(疊圖)', fontsize=9)
axes[1,0].axis('off')

# 後面每欄：單一 head
for h in range(n_heads):
    axes[0, h+1].imshow(attn_heads[h], cmap='inferno')
    axes[0, h+1].set_title(f'Head {h+1}', fontsize=9)
    axes[0, h+1].axis('off')
    axes[1, h+1].imshow(pet_img.resize((ATTN_SIZE, ATTN_SIZE)))
    axes[1, h+1].imshow(
        np.kron(attn_heads[h], np.ones((ATTN_SIZE//attn_heads.shape[1], ATTN_SIZE//attn_heads.shape[2]))),
        cmap='jet', alpha=0.5)
    axes[1, h+1].axis('off')

plt.suptitle(
    '[CLS] → patch attention：不同 head 關注不同部位（眼睛、輪廓、紋理…）\n上排=純 attention，下排=疊在原圖上',
    fontsize=11)
plt.tight_layout(); plt.show()

**觀察：**
- 不同 head 學到關注不同的視覺特徵——眼鼻嘴、毛皮邊緣、身體輪廓——這就是 multi-head attention 「分工」的證據
- 整體 attention 在主體上集中，背景幾乎沒被關注
- **這完全是 emergent 行為**——沒有任何 segmentation 標註訓練過 DINO

## 第 12 章：有 / 無 Register Tokens 對比

回顧 Part I 第 3 章：DINOv2 原版（無 registers）的 attention map 有時候會出現**高激活雜訊點**。我們現在親自看。

步驟：
1. 載入無 registers 版的 `facebook/dinov2-small`
2. 對同一張寵物圖跑同一個 attention 抽取邏輯
3. 並排對比

In [ ]:
MODEL_NAME_NOREG = 'facebook/dinov2-small'
# 同樣用 eager attention 以便拿 attention weights
model_noreg = AutoModel.from_pretrained(MODEL_NAME_NOREG, attn_implementation='eager').to(DEVICE).eval()
print(f'無 register 版: {MODEL_NAME_NOREG}')
print(f'  num_register_tokens = {getattr(model_noreg.config, "num_register_tokens", 0)}')

In [ ]:
# 抽取兩個模型的 attention
attn_reg   = get_attention_map(pet_img, model,       DEVICE).mean(0)   # 平均所有 head
attn_noreg = get_attention_map(pet_img, model_noreg, DEVICE).mean(0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
axes[0].imshow(pet_img.resize((ATTN_SIZE, ATTN_SIZE))); axes[0].set_title('原圖 / Original', fontsize=10); axes[0].axis('off')

axes[1].imshow(attn_noreg, cmap='inferno')
axes[1].set_title('無 registers (dinov2-small)\n注意右上的雜訊高激活點', fontsize=10, color='#922B21')
axes[1].axis('off')

axes[2].imshow(attn_reg, cmap='inferno')
axes[2].set_title('有 registers (with-registers-small)\nattention 更乾淨集中', fontsize=10, color='#1E8449')
axes[2].axis('off')

# 差異圖
diff = attn_reg - attn_noreg
axes[3].imshow(diff, cmap='RdBu_r', vmin=-abs(diff).max(), vmax=abs(diff).max())
axes[3].set_title('差異 (有 - 無)\n紅=registers 加強處\n藍=registers 減弱處（多半是 artifact）', fontsize=9)
axes[3].axis('off')

plt.suptitle('Register tokens 為何重要：消除 attention artifact', fontsize=12)
plt.tight_layout(); plt.show()

**觀察**（不同圖效果不同，可以多換幾張試試）：

- 無 register 版本常在背景區域出現幾個**不合理的亮點**——那就是「Vision Transformers Need Registers」論文指出的 artifact
- 有 register 版本 attention 更平滑、更聚焦在主體上
- 差異圖（最右）負值（藍色）多半出現在 artifact 位置——register tokens 把這些「計算暫存」吸走了

### 為什麼 artifact 會出現？

簡化解釋：
- ViT 在深層需要「全局計算暫存空間」（例如把整圖摘要傳給後續處理）
- 沒有 register 時，模型沒辦法，只能徵用某些不重要的 patch token 當暫存
- 那些 patch 的 attention 就被「污染」成異常高的值
- 加 register token = 給模型專門的 scratchpad → 不再徵用 patch tokens → attention 乾淨

---
# Part V — 收束 / Wrap-up

## 第 13 章：總結 + 延伸方向

### 我們完成了什麼

| Part | 內容 | 學到 |
|---|---|---|
| **I.** 直覺 | DINO 演算法、SSL motivation、v1→v3 演進 | **為什麼** DINO works |
| **II.** 解剖 | 載入 DINOv2、拆 token 結構、forward pass | 模型內部**長什麼樣** |
| **III.** 應用 | Feature extraction、kNN、Linear Probe、混淆分析 | DINOv2 features **多有用**（量化證據） |
| **IV.** 證據 | Multi-head attention、registers vs 無 registers | DINOv2 features **為什麼有用**（視覺證據） |

### 核心 Take-aways

1. **Pretrained > from scratch**：以 small models 對比，frozen DINOv2 + kNN 通常打敗從零訓練的 ViT。實務上 99% 場合用 pretrained 不要從零訓
2. **[CLS] token 不是萬靈丹**：對 image-level task 用 CLS、對 dense task 用 patch tokens、register tokens 別當 feature 用
3. **Attention map 不只是 debug 工具**：DINO 系列的 attention 直接可用於 unsupervised object discovery、video segmentation 等下游任務（LOST、TokenCut 等論文）
4. **Linear Probe 是 SSL 黃金標準**：當你看任何新的 SSL 方法 paper，第一個看的數字就是 ImageNet linear probe top-1

### 延伸方向

- **Fine-tune 整個 backbone**：解凍全部參數、用較低 LR（e.g. 1e-5）對特定資料集精修，準確率通常再多 1-3%。代價：需要 GPU、過擬合風險
- **LoRA / Adapter**：凍結 backbone，插入低秩 adapter（PEFT 框架），保留 generalization 又能微調
- **DINOv3**：2025 最新版，patch size 回到 16，加入 gated attention 與 Gram anchoring，效能再升一階
- **PCA 視覺化 patch features**：DINOv2 paper 經典圖——對 patch features 做 PCA，前 3 components 當 RGB 渲染。相同物體部位會被塗成同色、跨圖一致（unsupervised semantic correspondence）
- **下游 dense task**：拿 patch tokens（跳過 CLS 和 registers）接 segmentation head、depth estimation head
- **檢索系統**：把所有圖的 features cache 起來、用 cosine similarity 做圖像檢索

### 給自己的小練習

1. 把第 7 章的 `pool_strategy` 改成 `'mean_patch'` 或 `'cls_mean'`，看 kNN / linear probe 準確率怎麼變
2. 把 ATTN_SIZE 從 518 改成 224 或 980，比較 attention map 細緻度
3. 載入 `facebook/dinov2-with-registers-base`，比較和 small 的差異
4. 拿一張**非寵物**的圖（風景、產品、人臉）跑 attention map——還會找到主體嗎？